#Setup

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DoubleType, TimestampType
from pyspark.sql.window import Window

bronze_db = "workspace.bronze_restaurant"
silver_db = "workspace.silver_restaurant"

spark.sql(f"CREATE DATABASE IF NOT EXISTS {silver_db}")
print("Silver database ready.")

#Clean weather

In [0]:
df_weather = spark.table(f"{bronze_db}.weather")
df_weather.printSchema()
df_weather.show(5)

In [0]:
temps_min_max = df_weather.agg(
    F.min("temp").alias("temp_min"),
    F.max("temp").alias("temp_max")
)

display(temps_min_max)

In [0]:
from pyspark.sql.functions import col, sum

df_weather.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df_weather.columns
]).show()

In [0]:
#checking for mistakes in typos in "condition"
df_weather.select("condition").distinct().show()

In [0]:
from pyspark.sql.functions import col, count

# checking for duplicates
duplicates = df_weather.groupBy("weather_id", "timestamp") \
               .count() \
               .filter(col("count") > 1)

duplicates.show()

In [0]:
df_weather.write.format("delta").mode("overwrite") \
          .option("overwriteSchema", "true") \
          .saveAsTable(f"{silver_db}.weather")

#Clean orders

In [0]:
df_orders = spark.table(f"{bronze_db}.orders")
df_orders.printSchema()
df_orders.show(5)

In [0]:
df_orders.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df_orders.columns
]).show()

In [0]:
from pyspark.sql import Window

df_orders = df_orders.fillna({"server_id": 14})

window_spec = Window.partitionBy("timestamp").orderBy("order_id").rowsBetween(Window.currentRow, Window.unboundedFollowing)


df_orders = df_orders.withColumn(
    "timestamp",
    F.coalesce(
        F.col("timestamp"), 
        F.first("timestamp", ignorenulls=True).over(window_spec)
    )
)

In [0]:
df_orders.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df_orders.columns
]).show()

In [0]:
df_orders.select("order_type").distinct().show()

In [0]:
from pyspark.sql.functions import when, lower, trim

df_orders = df_orders.withColumn(
    "order_type",
    when(lower(trim(col("order_type"))).isin("to go", "to-go"), "to go")
    .when(lower(trim(col("order_type"))).isin("on site", "on-site", "onsite"), "on site")
    .when(lower(trim(col("order_type"))).isin("delivery", "delivrey"), "delivery")
    .otherwise("unknown")
)

In [0]:
df_orders.select("order_type").distinct().show()

In [0]:
duplicates = df_orders.groupBy(df_orders.columns) \
               .count() \
               .filter(col("count") > 1)

duplicates.show()

In [0]:
duplicates = df_orders.groupBy("order_id") \
               .count() \
               .filter(col("count") > 1)

duplicates.show()

In [0]:
df_orders = df_orders.dropDuplicates()

In [0]:
df_orders.show(5)

In [0]:
df_employees = spark.table(f"{bronze_db}.employees")

id_orphelins = df_orders.join(df_employees, df_orders.server_id == df_employees.emp_id, how="left_anti")

display(id_orphelins.select("server_id").distinct())

In [0]:
id_orphelins = df_orders.join(df_weather, on="weather_id", how="left_anti")

display(id_orphelins.select("weather_id").distinct())

In [0]:
df_orders.write.format("delta").mode("overwrite") \
         .option("overwriteSchema", "true") \
         .saveAsTable(f"{silver_db}.orders")

#Clean details

In [0]:
df_details = spark.table(f"{bronze_db}.order_details")
df_details.printSchema()
df_details.show(5)

In [0]:
qty_min_max = df_details.agg(
    F.min("qty").alias("qty_minimum"),
    F.max("qty").alias("qty_maximum")
)

display(qty_min_max)

In [0]:
df_details.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df_details.columns
]).show()

In [0]:
df_details = df_details.withColumn("qty", when(col("qty").isNull(), 1).otherwise(col("qty")))

In [0]:
duplicates = df_details.groupBy(df_details.columns) \
               .count() \
               .filter(col("count") > 1)

duplicates.show()

In [0]:
duplicates = df_details.groupBy("detail_id").count().filter(col("count") > 1)

duplicates.show()

In [0]:
df_details = df_details.dropDuplicates()
df_details.show(5)

In [0]:
df_menu = spark.table(f"{bronze_db}.menu")

id_orphelins = df_details.join(df_menu, on="item_id", how="left_anti")

display(id_orphelins.select("item_id").distinct())

In [0]:
id_orphelins = df_details.join(df_orders, on="order_id", how="left_anti")

display(id_orphelins.select("order_id").distinct())

In [0]:
df_details.write.format("delta").mode("overwrite") \
         .option("overwriteSchema", "true") \
         .saveAsTable(f"{silver_db}.order_details")